# 06. Query-based: RT-DETR

1. **Queries vs scene density:** are 300 queries enough for our frames?
2. **Sweep on fold 0** under the same compute budget as the CNN baseline: default, learning rate, denoising.
3. **Training dynamics:** held-out mAP against training hours, RT-DETR vs the YOLO baseline with the same budget.
4. **Small objects and dense scenes**, full cross-validation of the best experiment, submission.

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import pandas as pd

from core.cv import compare, cv_table, environment, existing, run_cv
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.evaluate import coco_evaluate
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.yolo import make_yolo_fns, plot_budget_curves

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames["objects"] = frames["uid"].map(lambda u: len(targets[u][0]))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## Queries vs scene density

In [ ]:
frames["objects"].describe(percentiles=[0.5, 0.95, 0.99]).round(1).to_frame("objects per frame").T.assign(
    share_over_300=(frames["objects"] > 300).mean())

## How bipartite matching removes NMS
During training, the Hungarian algorithm matches each ground-truth object to exactly one query (cost =
class score + L1 + GIoU of the boxes); every other query is trained to predict "no object". Since one object
can only ever be claimed by one query, the decoder learns, through self-attention between queries, to suppress
its own duplicates, and no NMS is needed at inference.

This changes training dynamics: the assignment of queries to objects is unstable early on (a query matched
to one object in one step may get another in the next), so DETRs converge slower than CNN detectors with fixed
anchors or locations. RT-DETR shortens this with query selection from encoder features and denoising queries:
noised ground-truth boxes the decoder learns to reconstruct, which gives a stable learning signal while the
matching is still noisy.

In [ ]:
BUDGET_HOURS = 1.5
BASE = dict(model="rtdetr-l.pt", imgsz=1280, epochs=100, batch=4, train_step=5, val_step=10,
            conf=0.001, iou=0.7, max_det=300, mask_ignore=True,
            train_args={"time": BUDGET_HOURS, "optimizer": "AdamW", "lr0": 1e-4, "weight_decay": 1e-4,
                        "cos_lr": True, "close_mosaic": 2})
EXPERIMENTS = {
    "rtdetr_default": BASE,
    "rtdetr_lr3e4": {**BASE, "train_args": {**BASE["train_args"], "lr0": 3e-4}},
    "rtdetr_no_denoising": {**BASE, "denoising": 0},
}
CNN = ["baseline"]


def config(name):
    return {**EXPERIMENTS[name], "name": name}

## Sweep on fold 0

In [ ]:
for name in EXPERIMENTS:
    print(name)
    run_cv(frames, gt, config(name), *make_yolo_fns(config(name), WORK, targets), WORK, folds=[0])

### Fold-0 comparison and training dynamics
mAP against training hours on the same held-out frames. The baseline trains without masking ignored regions.

In [ ]:
sweep = compare([*EXPERIMENTS, *CNN], WORK, fold=0)
display(sweep.round(4))
plot_budget_curves([*EXPERIMENTS, *CNN], 0, WORK);

### Small objects and dense scenes (fold 0)
mAP on frames with more than 100 objects vs the rest, next to AP_small from the table above.

In [ ]:
held0 = frames[frames["fold"] == 0]
subsets = {"dense (>100 objects)": held0[held0["objects"] > 100], "other": held0[held0["objects"] <= 100]}
rows = {}
for name in [*EXPERIMENTS, *CNN]:
    f = existing(f"oof_{name}_fold0.parquet", WORK)
    if f is not None:
        dets = pd.read_parquet(f)
        rows[name] = {s: coco_evaluate(dets, fr, gt)["mAP"] for s, fr in subsets.items()}
pd.DataFrame(rows).T.assign(frames_dense=len(subsets["dense (>100 objects)"])).round(4)

## Full cross-validation of the best experiment
Chosen on fold 0, so its fold-0 score is slightly optimistic.

In [ ]:
BEST = sweep.loc[sweep.index.isin(EXPERIMENTS), "mAP"].idxmax()
print("best on fold 0:", BEST)
_, predict_fn = make_yolo_fns(config(BEST), WORK, targets)
result = run_cv(frames, gt, config(BEST), *make_yolo_fns(config(BEST), WORK, targets), WORK)

In [ ]:
display(compare([BEST, *CNN], WORK).round(4))  # out-of-fold over all frames
cv_table(result).round(4)

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
test_dets = fold_ensemble(BEST, predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{BEST}.csv").head()